# ErrorInputs — malformed inputs fail cleanly

`MagneticStandalone3D` · ASN 28 negative (diagnostic) test case (TRUST-LiMe).


## Introduction

Negative (diagnostic) cases of the **standalone magnetic (induction) equation** added by TRUST-LiMe.

## What it validates

This notebook runs the category's three malformed-input datasets; each must
terminate with a non-zero exit code and a readable diagnostic, rather than
crashing silently:

- `error_missing_eta` — a missing magnetic diffusivity is rejected;
- `error_bad_nbcomposantes` — an invalid component count is rejected;
- `error_bad_bc` — an invalid boundary condition is rejected;

Each case is run **directly** with the `trust` command — not through
`runCases()`, which would abort the notebook on the non-zero exit.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 11 | malformed input fails cleanly | non-zero exit + readable message |

No mesh-convergence scan for this form: the diagnostic abort is mesh-independent.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Run


In [ ]:
import subprocess, tempfile, shutil, os

CASES = [
    ("error_missing_eta", "missing magnetic diffusivity"),
    ("error_bad_nbcomposantes", "invalid component count"),
    ("error_bad_bc", "invalid boundary condition"),
]
results = []
for CASE, what in CASES:
    tmp = tempfile.mkdtemp(prefix=f"lime_{CASE}_")
    shutil.copy(os.path.join("src", f"{CASE}.data"), tmp)
    try:
        p = subprocess.run(["trust", CASE, "2"], cwd=tmp,
                           capture_output=True, text=True, timeout=300)
        rc, out = p.returncode, (p.stdout + p.stderr)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    # a clean TRUST failure prints a diagnostic and exits non-zero
    assert rc != 0, f"{CASE}: expected a non-zero exit, got {rc}"
    assert out.strip(), f"{CASE}: no diagnostic message emitted"
    tail = [l for l in out.splitlines() if l.strip()][-1:] or [""]
    print(f"{CASE}: rc={rc}  |  {tail[0][:90]}")
    print("OK: malformed input rejected with a readable message.")
    results.append((CASE, rc))


## Result


In [ ]:
# Negative (diagnostic) cases: run directly with `trust`, so there is no
# trustutils suite and no performance table. Report the outcome instead.
print("Negative cases complete: all malformed inputs rejected with a non-zero exit.")
for CASE, rc in results:
    print(f"case: {CASE}  rc={rc}")
